# Multi-Asset Sentiment-Augmented DRL Trading
## Assets: BTC-USD | ETH-USD | NVDA | META
```
Why these 4 assets beat TSLA for DRL:
  BTC-USD : Extreme volatility, frequent 30-50% crashes, pure sentiment-driven
  ETH-USD : Even more volatile, DeFi news-driven
  NVDA    : AI boom stock, huge sentiment swings on chip news
  META    : -65% crash in 2022 val → agent MUST learn to exit

All anti-overfitting fixes from v3 applied:
  - Val: 2019-2022 (bear market)
  - Cost=True in ALL eval
  - Checkpoint rejection if val_CR > 500%
  - FFNN (no LSTM), dropout=0.3, orthogonal init
  - NaN guards everywhere
  - HP tuning per asset
```
### Before Running
1. Kaggle Secrets → HF_TOKEN | GPU T4 x1 | Internet ON
2. Cell 1 → Restart → Run All
3. Expected runtime: ~3-4 hrs total (all 4 assets)

In [ ]:
!pip install -q -U transformers peft datasets yfinance gymnasium scikit-learn torchao
import torch
print(f"Torch {torch.__version__} | CUDA {torch.cuda.is_available()}")
print("Restart kernel then Run All")

In [ ]:
# ── Asset configurations ──────────────────────────────────────
ASSETS_CONFIG = {
    "BTC-USD": {
        "keywords": ["bitcoin","btc","crypto","blockchain","cryptocurrency"],
        "name":     "Bitcoin (BTC)",
        "start":    "2015-01-01",   # BTC data from 2015
    },
    "ETH-USD": {
        "keywords": ["ethereum","eth","defi","blockchain","crypto"],
        "name":     "Ethereum (ETH)",
        "start":    "2016-01-01",   # ETH data from 2016
    },
    "NVDA": {
        "keywords": ["nvidia","nvda","gpu","jensen huang","h100","blackwell","ai chip"],
        "name":     "Nvidia (NVDA)",
        "start":    "2010-01-01",
    },
    "META": {
        "keywords": ["meta","facebook","zuckerberg","instagram","metaverse","reels"],
        "name":     "Meta (META)",
        "start":    "2012-06-01",   # META IPO was 2012
    },
}

# Shared settings
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_END  = "2018-12-31"
VAL_START  = "2019-01-01"
VAL_END    = "2022-12-31"
TEST_START = "2023-01-01"

MAX_EP_SEARCH    = 100   # per HP config (fast — 4 assets)
MAX_EP_FINAL     = 300   # final training per agent
EVAL_EVERY       = 25
PATIENCE         = 5
MAX_STEPS_PER_EP = 252
MAX_REALISTIC_VAL_CR = 500.0

REPO = "vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned"
BASE = "Qwen/Qwen2.5-3B-Instruct"

print(f"Device: {DEVICE}")
print(f"Assets: {list(ASSETS_CONFIG.keys())}")
print(f"Train: up to {TRAIN_END} | Val: {VAL_START}-{VAL_END} | Test: {TEST_START}+")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("HF login OK")

In [ ]:
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import gymnasium as gym
import yfinance as yf
import json, os, gc, copy, hashlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading SEntFiN Qwen2.5-3B...")
sent_tok   = AutoTokenizer.from_pretrained(REPO, token=HF_TOKEN)
sent_base  = AutoModelForCausalLM.from_pretrained(
    BASE, dtype=torch.float16, device_map="auto", token=HF_TOKEN)
sent_model = PeftModel.from_pretrained(sent_base, REPO, token=HF_TOKEN)
sent_model.eval()

# Pre-compute label token IDs
lids = {}
for lbl in ["positive","negative","neutral"]:
    ids = []
    for v in [lbl," "+lbl,lbl.capitalize()," "+lbl.capitalize()]:
        ids.extend(sent_tok.encode(v, add_special_tokens=False))
    lids[lbl] = list(set(ids))
n_pos = len(lids["positive"]); n_neg = len(lids["negative"])
all_ids = lids["positive"] + lids["negative"] + lids["neutral"]

def score_article(text, asset_name):
    SYSTEM = (f"You are a financial sentiment analyst. Classify the sentiment towards "
              f"{asset_name}. Reply with exactly one word: positive, negative, or neutral.")
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":f"Headline: {' '.join(text.split()[:300])}\nCompany: {asset_name}\nSentiment:"}]
    try: prompt = sent_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except: prompt = f"{SYSTEM}\n\nHeadline: {text[:300]}\nCompany: {asset_name}\nSentiment:"
    inp = sent_tok(prompt, return_tensors="pt", truncation=True, max_length=400).to(DEVICE)
    with torch.no_grad(): logits = sent_model(**inp).logits[0,-1,:]
    probs = torch.softmax(logits[all_ids], dim=0).cpu()
    return float(probs[:n_pos].sum() - probs[n_pos:n_pos+n_neg].sum())

print("✅ Scorer ready")

In [ ]:
import hashlib
from datasets import load_dataset

def load_prices(ticker, start):
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    df = df[["Close","Volume"]].copy(); df.columns = ["close","volume"]
    df.index = pd.to_datetime(df.index).normalize()
    return df.dropna().reset_index().rename(columns={"Date":"date"})

def safe_ts(ts):
    try:
        t = pd.Timestamp(ts) if not isinstance(ts, pd.Timestamp) else ts
        if t.tzinfo is not None: t = t.tz_localize(None)
        return t.normalize()
    except: return None

def is_relevant(text, keywords):
    t = (text or "").lower()
    return any(kw in t for kw in keywords)

def load_and_score_sentiment(ticker, cfg):
    """Load news, score with SEntFiN, return daily sentiment Series."""
    CACHE = Path(f"/kaggle/working/sent_cache_{ticker}.json")
    CSV   = Path(f"/kaggle/working/daily_sentiment_{ticker}.csv")

    if CSV.exists():
        print(f"  [{ticker}] Loading CSV (instant)...")
        saved = pd.read_csv(CSV, index_col=0, parse_dates=True)
        saved.index = pd.to_datetime(saved.index).tz_localize(None)
        return saved["score"]

    cache = {}
    if CACHE.exists():
        with open(CACHE) as f: cache = json.load(f)

    keywords = cfg["keywords"]
    all_articles = []
    MAX_N = 3000

    for ds_name, use_stream in [
        ("Brianferrell787/financial-news-multisource", True),
        ("ashraq/financial-news-articles", False),
        ("Johnson8187/us_economic_news_article", False),
    ]:
        try:
            ds = load_dataset(ds_name, split="train", streaming=use_stream)
            n = 0
            for ex in ds:
                t = ex.get("text") or ex.get("title") or ex.get("content") or ""
                if not is_relevant(t, keywords): continue
                d = safe_ts(ex.get("date") or ex.get("publishedAt") or "")
                if d is None: d = pd.Timestamp("2016-01-01") + pd.Timedelta(days=n*2)
                all_articles.append({"text": str(t)[:400], "date": d}); n += 1
                if n >= MAX_N: break
            print(f"  [{ticker}] {ds_name}: {n} articles")
        except Exception as e:
            print(f"  [{ticker}] {ds_name}: {e}")

    # yfinance recent news
    try:
        tick = yf.Ticker(ticker)
        try:    items = tick.get_news()
        except: items = getattr(tick, "news", []) or []
        for item in (items or []):
            title = item.get("title","") or item.get("content",{}).get("title","")
            ts    = item.get("providerPublishTime") or item.get("content",{}).get("pubDate","")
            if not title: continue
            d = safe_ts(pd.Timestamp(ts,unit="s") if isinstance(ts,(int,float)) else ts) if ts else None
            if d: all_articles.append({"text": title[:400], "date": d})
    except: pass

    print(f"  [{ticker}] Total: {len(all_articles)} articles — scoring...")
    daily_scores = {}
    n_new = 0

    for i, row in enumerate(all_articles):
        key   = hashlib.md5(row["text"][:500].encode()).hexdigest()
        score = cache.get(key)
        if score is None:
            score = score_article(row["text"], cfg["name"])
            cache[key] = score; n_new += 1
        daily_scores.setdefault(row["date"], []).append(score)
        if (i+1) % 200 == 0:
            with open(CACHE,"w") as f: json.dump(cache,f)

    with open(CACHE,"w") as f: json.dump(cache,f)
    daily_mean = {d: sum(v)/len(v) for d,v in daily_scores.items()}
    daily_sent = pd.Series(daily_mean).sort_index()
    out = daily_sent.reset_index(); out.columns = ["date","score"]
    out.to_csv(CSV, index=False)
    print(f"  [{ticker}] Saved {CSV} ({n_new} new scored)")
    return daily_sent

def build_states(df, sentiment, warmup=60):
    df = df.copy().sort_values("date").reset_index(drop=True)
    c, v = df["close"], df["volume"]
    ema10 = c.ewm(span=10,adjust=False).mean(); ema50 = c.ewm(span=50,adjust=False).mean()
    delta = c.diff(); gain = delta.clip(lower=0).ewm(span=14,adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(span=14,adjust=False).mean()
    rsi14 = 100 - (100/(1+gain/(loss+1e-10)))
    ema12 = c.ewm(span=12,adjust=False).mean(); ema26 = c.ewm(span=26,adjust=False).mean()
    macd  = ema12 - ema26; sig = macd.ewm(span=9,adjust=False).mean()
    sma20 = c.rolling(20).mean(); std20 = c.rolling(20).std()
    df["log_ret"]   = np.log(c/c.shift(1)+1e-10); df["daily_ret"] = c.pct_change()
    df["ema10_f"]   = ema10/c-1; df["ema50_f"] = ema50/c-1
    df["rsi_f"]     = rsi14/50-1; df["macd_f"] = (macd-sig)/(c+1e-10)
    df["boll_f"]    = (c-sma20)/(2*std20+1e-10); df["vol_f"] = np.log(v/v.shift(1)+1e-10)
    dates = pd.DatetimeIndex(df["date"])
    df["dow_sin"] = np.sin(2*np.pi*dates.dayofweek/7); df["dow_cos"] = np.cos(2*np.pi*dates.dayofweek/7)
    df["dom_sin"] = np.sin(2*np.pi*dates.day/31);      df["dom_cos"] = np.cos(2*np.pi*dates.day/31)
    df["moy_sin"] = np.sin(2*np.pi*dates.month/12);    df["moy_cos"] = np.cos(2*np.pi*dates.month/12)
    raw_sent = df["date"].map(dict(zip(sentiment.index,sentiment.values))).fillna(0.0)
    df["sentiment"] = raw_sent.ewm(span=7,adjust=False).mean()
    df = df.iloc[warmup:].dropna().reset_index(drop=True)
    for col in ["log_ret","ema10_f","ema50_f","rsi_f","macd_f","boll_f","vol_f"]:
        mu = df[col].rolling(252,min_periods=20).mean()
        sd = df[col].rolling(252,min_periods=20).std().clip(lower=1e-8)
        df[col] = (df[col]-mu)/sd
    S = np.zeros((len(df),15),np.float32)
    S[:,0]=df["log_ret"]; S[:,1]=0.0
    S[:,2]=df["dow_sin"]; S[:,3]=df["dow_cos"]
    S[:,4]=df["dom_sin"]; S[:,5]=df["dom_cos"]
    S[:,6]=df["moy_sin"]; S[:,7]=df["moy_cos"]
    S[:,8]=df["ema10_f"]; S[:,9]=df["ema50_f"]
    S[:,10]=df["rsi_f"];  S[:,11]=df["macd_f"]
    S[:,12]=df["boll_f"]; S[:,13]=df["vol_f"]
    S[:,14]=df["sentiment"]
    return np.nan_to_num(np.clip(S,-5,5),nan=0.0), df["daily_ret"].values.astype(np.float32)

print("Data + sentiment functions ready")

In [ ]:
# ── Trading Environment ───────────────────────────────────────
class TradingEnv(gym.Env):
    P={0:-1,1:0,2:1}
    def __init__(self,S,R,rand=True,cost=True):
        super().__init__()
        self.S=S.astype(np.float32); self.R=R.astype(np.float32)
        self.T=len(S); self.rand=rand; self.tc=0.002 if cost else 0.0
        self.action_space=gym.spaces.Discrete(3)
        self.observation_space=gym.spaces.Box(-np.inf,np.inf,(15,),np.float32)
        self.reset()
    def reset(self,seed=None,options=None):
        super().reset(seed=seed)
        self.t=int(np.random.randint(1,max(2,self.T-100))) if self.rand else 1
        self.val=100_000.0; self.prev=0; self.vals=[self.val]; self.acts=[0]
        return self._obs(),{}
    def _obs(self):
        s=self.S[self.t].copy(); s[1]=float(self.prev)
        return np.nan_to_num(s,nan=0.0)
    def step(self,action):
        pos=self.P[action]; ret=float(self.R[self.t])
        if not np.isfinite(ret): ret=0.0
        cost=self.tc if pos!=self.prev else 0.0
        nv=max(self.val*(1+pos*ret)*(1-cost),100.0)
        raw=np.log(nv/self.val+1e-10)-np.log(1+ret+1e-10)
        alpha_r=float(np.clip(raw,-0.03,0.03))
        self.val=nv; self.prev=pos; self.t+=1
        self.vals.append(nv); self.acts.append(pos)
        done=(self.t>=self.T-1)
        return (self._obs() if not done else np.zeros(15,np.float32)),alpha_r,done,False,{}
    def metrics(self):
        v=np.array(self.vals); r=np.diff(v)/v[:-1]
        sr=float(np.mean(r)/(np.std(r)+1e-10)*np.sqrt(252))
        cr=float((v[-1]/v[0]-1)*100)
        pk=np.maximum.accumulate(v); mdd=float(abs(((v-pk)/pk).min())*100)
        return {"CR%":round(cr,2),"SR":round(sr,3),"MDD%":round(mdd,2)}

def make_env(S,R,rand=True,cost=True): return TradingEnv(S,R,rand,cost)

# ── Networks (NaN-safe: no LayerNorm, orthogonal init) ─────────
def make_ffnn(inp,hid,out,nl=2,drop=0.3):
    lays=[nn.Linear(inp,hid),nn.ReLU(),nn.Dropout(drop)]
    for _ in range(nl-1): lays+=[nn.Linear(hid,hid),nn.ReLU(),nn.Dropout(drop)]
    lays.append(nn.Linear(hid,out))
    net=nn.Sequential(*lays)
    for m in net.modules():
        if isinstance(m,nn.Linear): nn.init.orthogonal_(m.weight,gain=0.01); nn.init.zeros_(m.bias)
    return net

class QNet(nn.Module):
    def __init__(self,hid=128,nl=2): super().__init__(); self.net=make_ffnn(15,hid,3,nl,0.2)
    def forward(self,x): return self.net(x)

class DDPGActor(nn.Module):
    def __init__(self,hid=128,nl=2):
        super().__init__()
        lays=[nn.Linear(15,hid),nn.ReLU(),nn.Dropout(0.2)]
        for _ in range(nl-1): lays+=[nn.Linear(hid,hid),nn.ReLU(),nn.Dropout(0.2)]
        lays+=[nn.Linear(hid,1),nn.Tanh()]; self.net=nn.Sequential(*lays)
        for m in self.net.modules():
            if isinstance(m,nn.Linear): nn.init.orthogonal_(m.weight,gain=0.01); nn.init.zeros_(m.bias)
    def forward(self,x): return self.net(x)
    def discrete(self,x):
        with torch.no_grad(): v=self.net(x.unsqueeze(0)).item()
        if not np.isfinite(v): v=0.0
        return (0 if v<-1/3 else (2 if v>1/3 else 1)),v

class DDPGCritic(nn.Module):
    def __init__(self,hid=128,nl=2): super().__init__(); self.net=make_ffnn(16,hid,1,nl,0.2)
    def forward(self,s,a): return self.net(torch.cat([s,a],dim=-1))

class ReplayBuffer:
    def __init__(self,cap=10000):
        self.cap=cap; self.ptr=0; self.sz=0
        self.S=np.zeros((cap,15),np.float32); self.A=np.zeros(cap,np.int64)
        self.R=np.zeros(cap,np.float32); self.NS=np.zeros((cap,15),np.float32)
        self.D=np.zeros(cap,np.float32); self.CA=np.zeros(cap,np.float32)
    def push(self,s,a,r,ns,d,ca=0.0):
        i=self.ptr
        self.S[i]=s; self.A[i]=a; self.R[i]=r; self.NS[i]=ns; self.D[i]=d; self.CA[i]=ca
        self.ptr=(i+1)%self.cap; self.sz=min(self.sz+1,self.cap)
    def sample(self,bs,dev):
        i=np.random.choice(self.sz,bs,replace=False)
        return (torch.FloatTensor(self.S[i]).to(dev),torch.LongTensor(self.A[i]).to(dev),
                torch.FloatTensor(self.R[i]).to(dev),torch.FloatTensor(self.NS[i]).to(dev),
                torch.FloatTensor(self.D[i]).to(dev),torch.FloatTensor(self.CA[i]).to(dev))
    def __len__(self): return self.sz

class OUNoise:
    def __init__(self,th=0.15,sig=0.3): self.th=th; self.sig=sig; self.reset()
    def reset(self): self.s=0.0
    def __call__(self): self.s+=self.th*(0-self.s)+self.sig*np.random.randn(); return float(np.clip(self.s,-1,1))

# ── Agents (NaN-safe) ─────────────────────────────────────────
def _safe(s): return np.nan_to_num(np.array(s,dtype=np.float32),nan=0.0,posinf=5.0,neginf=-5.0)
def add_noise(obs,std=0.02): return _safe(obs)+np.random.normal(0,std,obs.shape).astype(np.float32)

class PGAgent:
    def __init__(self,hid=64,nl=1,lr=1e-3,gamma=0.99,ent=0.05,wd=1e-3):
        self.gamma=gamma; self.ent=ent
        self.policy=make_ffnn(15,hid,3,nl,0.3).to(DEVICE)
        self.opt=torch.optim.Adam(self.policy.parameters(),lr=lr,weight_decay=wd)
        self.lp=[]; self.rw=[]; self.en=[]
    def act(self,s):
        s=_safe(s); lg=self.policy(torch.FloatTensor(s).to(DEVICE))
        if torch.isnan(lg).any(): return np.random.randint(3)
        d=torch.distributions.Categorical(logits=lg); a=d.sample()
        self.lp.append(d.log_prob(a)); self.en.append(d.entropy()); return a.item()
    def store(self,r): self.rw.append(float(r) if np.isfinite(r) else 0.0)
    def update(self):
        if not self.rw: return {}
        G=0.0; ret=[]
        for r in reversed(self.rw): G=r+self.gamma*G; ret.insert(0,G)
        ret=torch.FloatTensor(ret).to(DEVICE)
        if torch.isnan(ret).any(): self.lp=[]; self.rw=[]; self.en=[]; return {}
        ret=(ret-ret.mean())/(ret.std()+1e-8)
        loss=-(torch.stack(self.lp)*ret).mean()-self.ent*torch.stack(self.en).mean()
        if torch.isnan(loss): self.lp=[]; self.rw=[]; self.en=[]; return {}
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(),0.5)
        self.opt.step(); self.lp=[]; self.rw=[]; self.en=[]; return {}
    def greedy(self,s):
        s=_safe(s); self.policy.eval()
        with torch.no_grad(): lg=self.policy(torch.FloatTensor(s).to(DEVICE))
        self.policy.train()
        if torch.isnan(lg).any(): return 1
        return lg.argmax().item()

class PPOAgent:
    def __init__(self,hid=64,nl=1,lr=3e-4,gamma=0.99,lam=0.95,eps=0.2,vc=0.5,ec=0.05,epochs=4,wd=1e-3):
        self.gamma=gamma; self.lam=lam; self.eps=eps; self.vc=vc; self.ec=ec; self.epochs=epochs
        self.backbone=nn.Sequential(nn.Linear(15,hid),nn.ReLU(),nn.Dropout(0.3)).to(DEVICE)
        self.actor=nn.Linear(hid,3).to(DEVICE); self.critic=nn.Linear(hid,1).to(DEVICE)
        for m in list(self.backbone.modules())+[self.actor,self.critic]:
            if isinstance(m,nn.Linear): nn.init.orthogonal_(m.weight,gain=0.01); nn.init.zeros_(m.bias)
        params=list(self.backbone.parameters())+list(self.actor.parameters())+list(self.critic.parameters())
        self.opt=torch.optim.Adam(params,lr=lr,weight_decay=wd)
        self.ss=[]; self.aa=[]; self.lp=[]; self.rr=[]; self.vv=[]; self.dd=[]
    def _fwd(self,x): h=self.backbone(x); return self.actor(h),self.critic(h).squeeze(-1)
    def act(self,s):
        s=_safe(s); st=torch.FloatTensor(s).to(DEVICE)
        with torch.no_grad(): lg,val=self._fwd(st)
        if torch.isnan(lg).any():
            self.ss.append(s); self.aa.append(1); self.lp.append(0.0); self.vv.append(0.0); return 1
        d=torch.distributions.Categorical(logits=lg); a=d.sample()
        self.ss.append(s); self.aa.append(a.item())
        self.lp.append(d.log_prob(a).item()); self.vv.append(float(val)); return a.item()
    def store(self,r,done): self.rr.append(float(r) if np.isfinite(r) else 0.0); self.dd.append(done)
    def update(self,last_val=0.0):
        if len(self.rr)<2:
            self.ss=[]; self.aa=[]; self.lp=[]; self.rr=[]; self.vv=[]; self.dd=[]; return {}
        vals=self.vv+[last_val]; adv=[]; g=0.0
        for t in reversed(range(len(self.rr))):
            mask=1-self.dd[t]; delta=self.rr[t]+self.gamma*vals[t+1]*mask-vals[t]
            g=delta+self.gamma*self.lam*mask*g; adv.insert(0,g)
        adv=torch.FloatTensor(adv).to(DEVICE); adv=(adv-adv.mean())/(adv.std()+1e-8)
        ret=adv+torch.FloatTensor(self.vv).to(DEVICE)
        st=torch.FloatTensor(np.array(self.ss)).to(DEVICE)
        at=torch.LongTensor(self.aa).to(DEVICE); olp=torch.FloatTensor(self.lp).to(DEVICE)
        all_p=list(self.backbone.parameters())+list(self.actor.parameters())+list(self.critic.parameters())
        for _ in range(self.epochs):
            lg,nv=self._fwd(st)
            if torch.isnan(lg).any(): break
            d=torch.distributions.Categorical(logits=lg); ratio=(d.log_prob(at)-olp).exp()
            al=-torch.min(ratio*adv,torch.clamp(ratio,1-self.eps,1+self.eps)*adv).mean()
            loss=al+self.vc*F.mse_loss(nv,ret)-self.ec*d.entropy().mean()
            if torch.isnan(loss): break
            self.opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(all_p,0.5); self.opt.step()
        self.ss=[]; self.aa=[]; self.lp=[]; self.rr=[]; self.vv=[]; self.dd=[]; return {}
    def greedy(self,s):
        s=_safe(s); self.backbone.eval(); self.actor.eval()
        with torch.no_grad(): lg,_=self._fwd(torch.FloatTensor(s).to(DEVICE))
        self.backbone.train(); self.actor.train()
        if torch.isnan(lg).any(): return 1
        return lg.argmax().item()

class DQLAgent:
    def __init__(self,hid=128,nl=2,lr=1e-3,gamma=0.99,buf=20000,bs=64,tgt=100,
                 eps_s=1.0,eps_e=0.05,eps_d=5000,wd=1e-3):
        self.gamma=gamma; self.bs=bs; self.tgt=tgt; self.steps=0
        self.eps=eps_s; self.eps_s=eps_s; self.eps_e=eps_e; self.eps_d=eps_d
        self.online=QNet(hid,nl).to(DEVICE); self.target=QNet(hid,nl).to(DEVICE)
        self.target.load_state_dict(self.online.state_dict()); self.target.eval()
        self.opt=torch.optim.Adam(self.online.parameters(),lr=lr,weight_decay=wd)
        self.buf=ReplayBuffer(buf)
    def act(self,s):
        s=_safe(s); self.eps=max(self.eps_e,self.eps_s-self.steps/self.eps_d*(self.eps_s-self.eps_e))
        if np.random.rand()<self.eps: return np.random.randint(3)
        self.online.eval()
        with torch.no_grad(): q=self.online(torch.FloatTensor(s).unsqueeze(0).to(DEVICE))
        self.online.train()
        if torch.isnan(q).any(): return np.random.randint(3)
        return q.argmax().item()
    def store(self,s,a,r,ns,d):
        self.buf.push(_safe(s),a,float(r) if np.isfinite(r) else 0.0,_safe(ns),d); self.steps+=1
    def update(self):
        if len(self.buf)<max(self.bs,1000): return {}
        s,a,r,ns,d,_=self.buf.sample(self.bs,DEVICE)
        with torch.no_grad():
            na=self.online(ns).argmax(1,keepdim=True); tq=self.target(ns).gather(1,na).squeeze()
            tgt=r+self.gamma*tq*(1-d)
        cq=self.online(s).gather(1,a.unsqueeze(1)).squeeze(); loss=F.smooth_l1_loss(cq,tgt)
        if torch.isnan(loss): return {}
        self.opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(self.online.parameters(),0.5); self.opt.step()
        if self.steps%self.tgt==0: self.target.load_state_dict(self.online.state_dict())
        return {}
    def greedy(self,s):
        s=_safe(s); self.online.eval()
        with torch.no_grad(): q=self.online(torch.FloatTensor(s).unsqueeze(0).to(DEVICE))
        self.online.train()
        if torch.isnan(q).any(): return 1
        return q.argmax().item()

class DDPGAgent:
    def __init__(self,hid=128,nl=2,la=1e-4,lc=1e-3,gamma=0.99,tau=0.005,
                 buf=20000,bs=64,ou_t=0.15,ou_s=0.3,wd=1e-3):
        self.gamma=gamma; self.tau=tau; self.bs=bs
        self.actor=DDPGActor(hid,nl).to(DEVICE); self.critic=DDPGCritic(hid,nl).to(DEVICE)
        self.actor_t=DDPGActor(hid,nl).to(DEVICE); self.critic_t=DDPGCritic(hid,nl).to(DEVICE)
        self.actor_t.load_state_dict(self.actor.state_dict()); self.critic_t.load_state_dict(self.critic.state_dict())
        self.actor_t.eval(); self.critic_t.eval()
        self.opt_a=torch.optim.Adam(self.actor.parameters(),lr=la,weight_decay=wd)
        self.opt_c=torch.optim.Adam(self.critic.parameters(),lr=lc,weight_decay=wd)
        self.buf=ReplayBuffer(buf); self.noise=OUNoise(ou_t,ou_s)
    def _soft(self,o,t):
        for p,pt in zip(o.parameters(),t.parameters()): pt.data.copy_(self.tau*p.data+(1-self.tau)*pt.data)
    def act(self,s,explore=True):
        s=_safe(s); st=torch.FloatTensor(s).to(DEVICE)
        self.actor.eval(); disc,ca=self.actor.discrete(st); self.actor.train()
        if explore: ca=float(np.clip(ca+self.noise(),-1,1)); disc=0 if ca<-1/3 else (2 if ca>1/3 else 1)
        return disc,ca
    def store(self,s,a,r,ns,d,ca):
        self.buf.push(_safe(s),a,float(r) if np.isfinite(r) else 0.0,_safe(ns),d,float(ca))
    def update(self):
        if len(self.buf)<max(self.bs,1000): return {}
        s,a,r,ns,d,ca=self.buf.sample(self.bs,DEVICE)
        with torch.no_grad():
            na=self.actor_t(ns); tq=self.critic_t(ns,na).squeeze(); tgt=r+self.gamma*tq*(1-d)
        cq=self.critic(s,ca.unsqueeze(1)).squeeze(); cl=F.mse_loss(cq,tgt)
        if torch.isnan(cl): return {}
        self.opt_c.zero_grad(); cl.backward(); nn.utils.clip_grad_norm_(self.critic.parameters(),0.5); self.opt_c.step()
        pa=self.actor(s); al=-self.critic(s,pa).mean()
        if torch.isnan(al): return {}
        self.opt_a.zero_grad(); al.backward(); nn.utils.clip_grad_norm_(self.actor.parameters(),0.5); self.opt_a.step()
        self._soft(self.actor,self.actor_t); self._soft(self.critic,self.critic_t); return {}
    def greedy(self,s): disc,_=self.act(s,explore=False); return disc

print("Environment + Networks + Agents ready")

In [ ]:
def sharpe(vals):
    r=np.diff(np.array(vals))/np.array(vals[:-1])
    return float(np.mean(r)/(np.std(r)+1e-10)*np.sqrt(252)) if len(r)>1 else 0.0

def run_eval(agent,S,R):
    env=make_env(S,R,rand=False,cost=True)
    obs,_=env.reset(); done=False
    while not done: obs,_,done,_,_=env.step(agent.greedy(obs))
    return sharpe(env.vals),env.metrics()

def create_agent(name,cfg):
    if name=="PG":   return PGAgent(**cfg)
    elif name=="PPO": return PPOAgent(**cfg)
    elif name=="DQL": return DQLAgent(**cfg)
    elif name=="DDPG": return DDPGAgent(**cfg)

def save_state(name,agent):
    if name=="PG":   return copy.deepcopy(agent.policy.state_dict())
    elif name=="PPO": return (copy.deepcopy(agent.backbone.state_dict()),
                              copy.deepcopy(agent.actor.state_dict()),
                              copy.deepcopy(agent.critic.state_dict()))
    elif name=="DQL": return copy.deepcopy(agent.online.state_dict())
    elif name=="DDPG": return (copy.deepcopy(agent.actor.state_dict()),
                               copy.deepcopy(agent.critic.state_dict()))

def load_state(name,agent,st):
    if name=="PG":   agent.policy.load_state_dict(st)
    elif name=="PPO": agent.backbone.load_state_dict(st[0]); agent.actor.load_state_dict(st[1]); agent.critic.load_state_dict(st[2])
    elif name=="DQL": agent.online.load_state_dict(st)
    elif name=="DDPG": agent.actor.load_state_dict(st[0]); agent.critic.load_state_dict(st[1])

def run_episode(agent,name,env,trS,trR):
    obs,_=env.reset(); done=False; tot=0.0; steps=0; lv=0.0
    if name=="PG":
        while not done:
            a=agent.act(add_noise(obs)); obs,r,done,_,_=env.step(a); agent.store(r); tot+=r
        agent.update()
    elif name=="PPO":
        while not done:
            a=agent.act(add_noise(obs)); obs,r,done,_,_=env.step(a); agent.store(r,done); tot+=r
        if not done:
            with torch.no_grad(): _,lv=agent._fwd(torch.FloatTensor(_safe(obs)).to(DEVICE)); lv=float(lv)
        agent.update(lv)
    elif name=="DQL":
        while not done and steps<MAX_STEPS_PER_EP:
            a=agent.act(add_noise(obs)); no,r,done,_,_=env.step(a)
            agent.store(obs,a,r,no,done); agent.update(); obs=no; tot+=r; steps+=1
    elif name=="DDPG":
        agent.noise.reset()
        while not done and steps<MAX_STEPS_PER_EP:
            a,ca=agent.act(add_noise(obs)); no,r,done,_,_=env.step(a)
            agent.store(obs,a,r,no,done,ca); agent.update(); obs=no; tot+=r; steps+=1
    return tot

def train_one_agent(name,cfg,max_ep,trS,trR,vlS,vlR,label=""):
    agent=create_agent(name,cfg)
    best_sr=-np.inf; best_st=None; best_cr=0.0; no_imp=0
    for ep in range(1,max_ep+1):
        env=make_env(trS,trR,rand=True,cost=True)
        try: tot=run_episode(agent,name,env,trS,trR)
        except Exception as e: continue
        if ep%EVAL_EVERY==0:
            try: sr,m=run_eval(agent,vlS,vlR)
            except: sr=0.0; m={"CR%":0,"SR":0,"MDD%":0}
            val_cr=m["CR%"]; ok=val_cr<MAX_REALISTIC_VAL_CR
            flag="OK" if ok else "SKIP"
            print(f"  [{name}{label}] ep={ep:4d} | val_SR={sr:.3f} | val_CR={val_cr:.1f}% [{flag}]")
            if sr>best_sr and ok:
                best_sr=sr; best_cr=val_cr; best_st=save_state(name,agent); no_imp=0
            else:
                no_imp+=1
                if no_imp>=PATIENCE: print(f"  [{name}{label}] Early stop"); break
    if best_st: load_state(name,agent,best_st)
    return agent,best_sr,best_cr

HP_GRID = {
    "PG":  [
        {"hid":64, "nl":1,"lr":1e-3,"gamma":0.99,"ent":0.05,"wd":1e-3},
        {"hid":128,"nl":2,"lr":5e-4,"gamma":0.99,"ent":0.1, "wd":1e-3},
    ],
    "PPO": [
        {"hid":64, "nl":1,"lr":3e-4,"gamma":0.99,"lam":0.95,"eps":0.2,"vc":0.5,"ec":0.05,"epochs":4,"wd":1e-3},
        {"hid":128,"nl":1,"lr":1e-4,"gamma":0.99,"lam":0.95,"eps":0.1,"vc":0.5,"ec":0.1,"epochs":4,"wd":1e-3},
    ],
    "DQL": [
        {"hid":128,"nl":2,"lr":1e-3,"gamma":0.99,"buf":20000,"wd":1e-3},
        {"hid":256,"nl":2,"lr":5e-4,"gamma":0.99,"buf":20000,"wd":1e-3},
    ],
    "DDPG":[
        {"hid":128,"nl":2,"la":1e-4,"lc":1e-3,"gamma":0.99,"tau":0.005,"buf":20000,"ou_s":0.3,"wd":1e-3},
        {"hid":256,"nl":2,"la":5e-5,"lc":5e-4,"gamma":0.99,"tau":0.005,"buf":20000,"ou_s":0.3,"wd":1e-3},
    ],
}

def run_asset(ticker, asset_cfg):
    """Full pipeline for one asset. Returns result dict."""
    print(f"\n{'='*65}")
    print(f"ASSET: {ticker} ({asset_cfg['name']})")
    print(f"{'='*65}")

    # 1. Prices
    prices=load_prices(ticker,asset_cfg["start"])
    prices["date"]=pd.to_datetime(prices["date"])
    tr=prices[prices["date"]<=TRAIN_END].copy()
    vl=prices[(prices["date"]>=VAL_START)&(prices["date"]<=VAL_END)].copy()
    te=prices[prices["date"]>=TEST_START].copy()
    if len(tr)<100 or len(vl)<50 or len(te)<50:
        print(f"  Not enough data for {ticker}"); return None
    bnh_te=(te["close"].iloc[-1]/te["close"].iloc[0]-1)*100
    print(f"  Train:{len(tr)} Val:{len(vl)} Test:{len(te)} | B&H test={bnh_te:.1f}%")

    # 2. Sentiment
    daily_sent=load_and_score_sentiment(ticker,asset_cfg)
    def align(df): return pd.Series(df["date"].map(daily_sent).fillna(0.0).tolist(),index=df["date"].tolist())
    tr_s=align(tr); vl_s=align(vl); te_s=align(te)

    # 3. Features
    WM=60
    vl_ext=pd.concat([tr.tail(WM),vl]).reset_index(drop=True)
    te_ext=pd.concat([vl.tail(WM),te]).reset_index(drop=True)
    all_s=pd.concat([tr_s,vl_s,te_s])
    trS,trR=build_states(tr,tr_s)
    vlS,vlR=build_states(vl_ext,all_s)
    teS,teR=build_states(te_ext,all_s)
    print(f"  States: Train={len(trS)} Val={len(vlS)} Test={len(teS)}")

    # 4. HP search + final training
    best_cfgs={}
    for name,configs in HP_GRID.items():
        results=[]
        for i,cfg in enumerate(configs):
            ag,vsr,vcr=train_one_agent(name,cfg,MAX_EP_SEARCH,trS,trR,vlS,vlR,f"-c{i+1}")
            results.append((vsr,cfg)); del ag; gc.collect(); torch.cuda.empty_cache()
        results.sort(key=lambda x:x[0],reverse=True)
        best_cfgs[name]=results[0][1]
        print(f"  [{name}] Best config selected: val_SR={results[0][0]:.3f}")

    trained={}; val_srs={}
    for name,cfg in best_cfgs.items():
        ag,vsr,vcr=train_one_agent(name,cfg,MAX_EP_FINAL,trS,trR,vlS,vlR)
        trained[name]=ag; val_srs[name]=vsr

    # 5. Test evaluation
    bnh=make_env(teS,teR,rand=False,cost=False); bnh.reset()
    for _ in range(bnh.T-1): bnh.step(2)
    bnh_sr=sharpe(bnh.vals); bnh_m=bnh.metrics()
    print(f"  B&H  | CR={bnh_m['CR%']:.1f}% SR={bnh_m['SR']:.3f} MDD={bnh_m['MDD%']:.1f}%")

    asset_rows=[{"Asset":ticker,"Model":"B&H",**bnh_m,"ValSR":"--","beat":False}]
    for name,ag in trained.items():
        env=make_env(teS,teR,rand=False,cost=True)
        obs,_=env.reset(); done=False
        while not done: obs,_,done,_,_=env.step(ag.greedy(obs))
        tsr=sharpe(env.vals); m=env.metrics()
        beat=m["CR%"]>bnh_m["CR%"]
        print(f"  {name:6s}| CR={m['CR%']:.1f}% SR={m['SR']:.3f} MDD={m['MDD%']:.1f}% {'BEAT' if beat else '----'} ValSR={val_srs[name]:.3f}")
        asset_rows.append({"Asset":ticker,"Model":name,**m,"ValSR":round(val_srs[name],3),"beat":beat})

    return asset_rows, bnh.vals, {n:[] for n in trained}, trained, teS, teR

print("Training functions ready")

In [ ]:
# ══════════════════════════════════════════════════════════════
# MAIN: Run all 4 assets
# ══════════════════════════════════════════════════════════════
all_results = []   # accumulate rows from all assets

for ticker, asset_cfg in ASSETS_CONFIG.items():
    try:
        result = run_asset(ticker, asset_cfg)
        if result is not None:
            asset_rows, bnh_vals, _, trained, teS, teR = result
            all_results.extend(asset_rows)
    except Exception as e:
        print(f"\n[{ticker}] FAILED: {e}")
        import traceback; traceback.print_exc()
    finally:
        gc.collect(); torch.cuda.empty_cache()

# Combined results table
df_all = pd.DataFrame(all_results)
df_all.to_csv("/kaggle/working/results_multi_asset.csv", index=False)
print("\n" + "="*70)
print("MULTI-ASSET FINAL RESULTS")
print("="*70)
print(df_all[["Asset","Model","CR%","SR","MDD%","ValSR","beat"]].to_string(index=False))

# Summary: which agents beat B&H per asset
print("\nSUMMARY — Agents that BEAT B&H:")
for ticker in ASSETS_CONFIG:
    asset_df = df_all[df_all["Asset"]==ticker]
    bnh_cr   = asset_df[asset_df["Model"]=="B&H"]["CR%"].values[0] if len(asset_df)>0 else 0
    winners  = asset_df[(asset_df["Model"]!="B&H") & (asset_df["beat"]==True)]["Model"].tolist()
    print(f"  {ticker:8s} B&H={bnh_cr:.1f}% | Beaten by: {winners if winners else 'none'}")

In [ ]:
# ── Multi-asset visualization ─────────────────────────────────
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()
CLR = {"B&H":"black","PG":"#1f77b4","PPO":"#ff7f0e","DQL":"#2ca02c","DDPG":"#d62728"}

for idx, ticker in enumerate(ASSETS_CONFIG.keys()):
    ax = axes[idx]
    asset_df = df_all[df_all["Asset"]==ticker]
    if len(asset_df)==0: continue

    models = asset_df["Model"].tolist()
    crs    = asset_df["CR%"].tolist()
    colors = [CLR.get(m,"gray") for m in models]

    bars = ax.bar(models, crs, color=colors, edgecolor="white", lw=1.5)
    for bar, val in zip(bars, crs):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"{val:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")

    bnh_cr = asset_df[asset_df["Model"]=="B&H"]["CR%"].values[0]
    ax.axhline(y=bnh_cr, color="black", ls="--", alpha=0.7, lw=2, label=f"B&H={bnh_cr:.1f}%")
    ax.set_title(f"{ticker} — Test CR% (cost=True)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Cumulative Return %"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis="y")

fig.suptitle("Multi-Asset DRL Trading\nQwen2.5-3B SEntFiN (F1=0.9047) | Val:2019-2022(bear) | Test:2023+",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/dashboard_multi_asset.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved dashboard_multi_asset.png")
print("Saved results_multi_asset.csv")
print("\nDone! Download both files from Output tab.")